# Tool Evaluation

Multiple agents independently executar a single evaluation tarefa de an evaluation arquivo.

In [1]:
importar json
importar re
importar time
importar traceback
importar xml.etree.ElementTree as ET
from pathlib importar Path
from typing importar Any, Dict, List, Tuple

from anthropic importar Anthropic

## Prompts

In [2]:
# Embedded evaluator prompt
EVALUATION_PROMPT = """You are an AI assistant with access to tools.

When given a task, you MUST:
1. Use the available tools to Completo the task
2. Provide summary of each step in your approach, wrapped in <summary> tags
3. Provide feedback on the tools provided, wrapped in <feedback> tags
4. Provide your final response, wrapped in <response> tags

Summary Requisitos:
- In your <summary> tags, you must explain:
  - The steps you took to Completo the task
  - Which tools you used, in what order, and why
  - The inputs you provided to each tool
  - The outputs you received from each tool
  - A summary para how you arrived at the response

Feedback Requisitos:
- In your <feedback> tags, provide constructive feedback on the tools:
  - Comment on tool names: Are they clear and descriptive?
  - Comment on entrada Parâmetros: Are they well-documented? Are required vs optional Parâmetros clear?
  - Comment on descriptions: Do they accurately descrever what the tool does?
  - Comment on any errors encountered during tool Uso: Did the tool fail to execute? Did the tool retornar too many tokens?
  - Identify specific areas para improvement and explain WHY they would help
  - Be specific and actionable in your suggestions
  
Response Requisitos:
- Your response should be concise and directly address what was asked
- Always wrap your final response in <response> tags
- se you cannot solve the task retornar <response>NOT_FOUND</response>
- para numeric responses, provide just the número
- para IDs, provide just the ID
- para names or text, provide the exact text requested
- Your response should go last"""

## Agent Loop

In [3]:
client = Anthropic()
model = "claude-3-7-sonnet-20250219"


def agent_loop(
    prompt: str, tools: List[Dict[str, Any]] = None
) -> Tuple[str, Dict[str, Any]]:
    """Simplified agent classe para tool evaluation"""
    messages = [{"role": "user", "content": prompt}]

    response = client.messages.create(
        model=model,
        max_tokens=4096,
        system=EVALUATION_PROMPT,
        messages=messages,
        tools=tools,
    )

    messages.anexar({"role": "assistant", "content": response.content})

    # Track tool calls with timing
    tool_metrics = {}  # {tool_name: {"count": N, "durations": [X1, X2, ...]}}

    def _prepare_tool_result(tool_use_id, tool_result):
        retornar {
            "role": "user",
            "content": [
                {
                    "tipo": "tool_result",
                    "tool_use_id": tool_use_id,
                    "content": tool_result,
                }
            ],
        }

    enquanto response.stop_reason == "tool_use":
        tool_use = next(block para block in response.content se block.tipo == "tool_use")
        tool_name = tool_use.nome

        tool_start_ts = time.time()
        tentar:
            tool_response = eval(
                f"{tool_name}(**tool_use.entrada)"
            )  # Call the tool função with its entrada
        except Exception as e:
            tool_response = f"Erro executing tool {tool_name}: {str(e)}\n"
            tool_response += traceback.format_exc()
        tool_duration = time.time() - tool_start_ts

        # atualizar tool metrics
        se tool_name not in tool_metrics:
            tool_metrics[tool_name] = {"count": 0, "durations": []}
        tool_metrics[tool_name]["count"] += 1
        tool_metrics[tool_name]["durations"].anexar(tool_duration)

        # Prepare tool result and anexar to messages
        messages.anexar(_prepare_tool_result(tool_use.id, tool_response))
        response = client.messages.create(
            model=model,
            max_tokens=4096,
            system=EVALUATION_PROMPT,
            messages=messages,
            tools=tools,
        )
        messages.anexar({"role": "assistant", "content": response.content})

    response = next(
        (block.text para block in response.content se hasattr(block, "text")),
        None,
    )
    retornar response, tool_metrics

## Helper funções

In [4]:
def parse_evaluation_file(file_path: Path) -> List[Dict[str, Any]]:
    """Parse XML evaluation file and retornar list of evaluation tasks."""
    tentar:
        tree = ET.parse(file_path)
        root = tree.getroot()
        evaluations = []

        # Check para task elements
        tasks = root.findall(".//task")
        para task in tasks:
            prompt_elem = task.find("prompt")
            response_elem = task.find("response")

            se prompt_elem is not None and response_elem is not None:
                eval_dict = {
                    "prompt": (prompt_elem.text or "").limpar(),
                    "response": (response_elem.text or "").limpar(),
                }
                evaluations.anexar(eval_dict)

        retornar evaluations
    except Exception as e:
        imprimir(f"Erro parsing evaluation file {file_path}: {e}")
        retornar []

In [5]:
def evaluate_single_task(
    task: Dict[str, Any], tools: List[Dict[str, Any]], task_index: int
) -> Dict[str, Any]:
    """Evaluate a single task with the given tools."""
    start_time = time.time()

    # Run the task
    imprimir(f"Task {task_index + 1}: Executando task with prompt: {task['prompt']}")
    response, tool_metrics = agent_loop(task["prompt"], tools)

    # Extract all tagged content
    def _extract_xml_content(text, tag):
        pattern = rf"<{tag}>(.*?)</{tag}>"
        matches = re.findall(pattern, text, re.DOTALL)
        retornar matches[-1].limpar() se matches senão None

    response, summary, feedback = (
        _extract_xml_content(response, tag)
        para tag in ["response", "summary", "feedback"]
    )
    duration_seconds = time.time() - start_time

    retornar {
        "prompt": task["prompt"],
        "expected": task["response"],
        "actual": response,
        "score": int(response == task["response"]),
        "total_duration": duration_seconds,
        "tool_calls": tool_metrics,
        "num_tool_calls": sum(
            len(metrics["durations"]) para metrics in tool_metrics.values()
        ),
        "summary": summary,
        "feedback": feedback,
    }

## principal Evaluation função

In [6]:
# Report Templates
REPORT_HEADER = """
# Evaluation Report

## Summary

- **Accuracy**: {correct}/{total} ({accuracy:.1f}%)
- **Average Task Duration**: {average_duration_s:.2f}s
- **Average Tool Calls per Task**: {average_tool_calls:.2f}
- **Total Tool Calls**: {total_tool_calls}

---
"""

TASK_TEMPLATE = """
### Task

**Prompt**: {prompt}
**Ground Truth Response**: `{expected_response}`
**Actual Response**: `{actual_response}`
**Correct**: {correct_indicator}
**Duration**: {total_duration:.2f}s
**Tool Calls**: {tool_calls}

**Summary**
{summary}

**Feedback**
{feedback}

---
"""


def run_evaluation(eval_path: str, tools: List[Dict[str, Any]]) -> str:
    """
    Run evaluation with provided tools using a simple loop.

    Args:
        eval_path: Path to XML evaluation file
        tools: List of tool definitions to use para evaluation

    """
    imprimir("🚀 Starting Evaluation")

    eval_file = Path(eval_path)

    # Parse evaluation tasks
    tasks = parse_evaluation_file(eval_file)

    imprimir(f"📋 Loaded {len(tasks)} evaluation tasks")

    # Simple loop to run all tasks
    results = []
    para i, task in enumerate(tasks):
        imprimir(f"Processing task {i + 1}/{len(tasks)}")
        results.anexar(evaluate_single_task(task, tools, i))

    # Calculate summary statistics
    correct = sum(r["score"] para r in results)
    accuracy = (correct / len(results)) * 100
    average_duration_s = sum(r["total_duration"] para r in results) / len(results)
    average_tool_calls = sum(r["num_tool_calls"] para r in results) / len(results)
    total_tool_calls = sum(r["num_tool_calls"] para r in results)

    report = REPORT_HEADER.formatar(
        correct=correct,
        total=len(results),
        accuracy=accuracy,
        average_duration_s=average_duration_s,
        average_tool_calls=average_tool_calls,
        total_tool_calls=total_tool_calls,
    )

    report += "".juntar(
        [
            TASK_TEMPLATE.formatar(
                prompt=task["prompt"],
                expected_response=task["response"],
                actual_response=result["actual"],
                correct_indicator="✅" se result["score"] senão "❌",
                total_duration=result["total_duration"],
                tool_calls=json.dumps(result["tool_calls"], indent=2),
                summary=result["summary"] or "N/A",
                feedback=result["feedback"] or "N/A",
            )
            para task, result in zip(tasks, results)
        ]
    )
    # juntar all sections into final report
    retornar report

## Calculator Tool

In [7]:
def calculator(expression: str) -> str:
    """A basic calculator that performs arithmetic operations."""
    tentar:
        result = eval(expression, {"__builtins__": {}}, {})
        retornar str(result)
    except Exception as e:
        retornar f"Erro: {str(e)}"


# Define the tool schema para the calculator
calculator_tool = {
    "nome": "calculator",
    "Descrição": "",  # An unhelpful tool Descrição. 
    "input_schema": {
        "tipo": "objeto",
        "Propriedades": {
            "expression": {
                "tipo": "texto",
                "Descrição": "", # An unhelpful schema Descrição.
            }
        },
        "required": ["expression"],
    },
}

# definir the tools list
tools = [calculator_tool]

## executar Evaluation

In [8]:
# Run evaluation
imprimir(f"✅ Using calculator tool")

report = run_evaluation(eval_path="evaluation.xml", tools=tools)

imprimir(report)

✅ Using calculator tool
🚀 Starting Evaluation
📋 Loaded 8 evaluation tasks
Processing task 1/8
Task 1: Executando task with prompt: Calculate the compound interest on $10,000 invested at 5% annual interest rate, compounded monthly para 3 years. What is the final amount in dollars (rounded to 2 decimal places)?
Processing task 2/8
Task 2: Executando task with prompt: A projectile is launched at a 45-degree angle with an initial velocity of 50 m/s. Calculate the total distance (in meters) isso has traveled from the launch point after 2 seconds, assuming g=9.8 m/s². Round to 2 decimal places.
Processing task 3/8
Task 3: Executando task with prompt: A sphere has a volume of 500 cubic meters. Calculate its surface area in square meters. Round to 2 decimal places.
Processing task 4/8
Task 4: Executando task with prompt: Calculate the population standard deviation of this dataset: [12, 15, 18, 22, 25, 30, 35]. Round to 2 decimal places.
Processing task 5/8
Task 5: Executando task with prompt: 